# Res:
- Baseline (0.56108)
- Baseline + LogMel (0.74745)
- Augs (0.73133)
- Augs v2 (Noises + RIRs) (0.75494)
- Focal Loss (0.73389)

# EAT:
- Augs V3
- Augs V3 + Focal loss
- Augs V3 + pass filters

In [ ]:
!uv pip install -q "torch==2.8.0" "torchvision==0.23.0" "torchaudio==2.8.0" --torch-backend=cu126
!uv pip install -q audiomentations efficientnet_pytorch "transformers<5.0.0"

In [ ]:
from collections import Counter
from enum import Enum
from pathlib import Path
from typing import override

import audiomentations as A
import librosa
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import soundfile as sf
import torch
import torchaudio
import torchaudio.transforms as T
import torchmetrics
from sklearn.model_selection import train_test_split
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils import data
from transformers import ASTFeatureExtractor, ASTModel, AutoModel

In [ ]:
class RunType(Enum):
    DATA = "data"
    BASELINE = "baseline"
    AUGS = "augs"
    AUGSv2 = "augs_v2"
    AUGSv3 = "augs_v3"
    AUGSv4 = "augs_v4"
    FOCAL = "focal"
    BASESSL = "base_ssl"
    AST = "ast"


CURR_RUN_TYPE = RunType("ast")
WAV_PATH = Path("/kaggle/input/competitions/itmo-acoustic-event-detectin-2026/")

METRIC = "val_f1"
MODE = "max"

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath=f"./checkpoints/{CURR_RUN_TYPE}",
    filename="aed-{epoch:02d}-{val_f1:.2f}",
    monitor=METRIC,
    mode=MODE,
    save_top_k=1,
    save_last=False,
)

early_stopping = pl.callbacks.EarlyStopping(
    monitor=METRIC,
    min_delta=0.02,
    patience=4,
    verbose=False,
    mode=MODE,
)

logger = pl.loggers.CSVLogger("./logs", name=f"{CURR_RUN_TYPE}_experiment")

trainer = pl.Trainer(
    callbacks=[checkpoint_callback, early_stopping],
    logger=[logger],
    max_epochs=30,
    accelerator="gpu",
    # devices=2,
    # strategy="ddp_notebook",
    enable_progress_bar=True,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
def get_pathes_labels_conv(df, label2idx: dict | None = None, pred: bool = False):
    if label2idx is None:
        label2idx = {label: idx for idx, label in enumerate(np.unique(df["label"]))}
    prefix = "audio_train/train" if not pred else "audio_test/test"
    pathes = df["fname"].apply(lambda fname: WAV_PATH.joinpath(prefix, fname)).to_list()
    if pred:
        return pathes, None, label2idx
    labels = df["label"].apply(lambda label: label2idx[label]).to_list()
    return pathes, labels, label2idx


df = pd.read_csv("/kaggle/input/competitions/itmo-acoustic-event-detectin-2026/train.csv")
pathes, labels, label2idx = get_pathes_labels_conv(df)
idx2label = {value: key for key, value in label2idx.items()}
df.head(5)

,fname,label
0,8bcbcc394ba64fe85ed4.wav,Finger_snapping
1,00d77b917e241afa06f1.wav,Squeak
2,17bb93b73b8e79234cb3.wav,Electric_piano
3,7d5c7a40a936136da55e.wav,Harmonica
4,17e0ee7565a33d6c2326.wav,Snare_drum


In [ ]:
if CURR_RUN_TYPE == RunType.DATA:
    df["length"] = df["fname"].apply(lambda path: len(sf.read(WAV_PATH.joinpath(path))[0]))
    print(df["length"].describe())
    print(df["label"].value_counts())

# Baseline

## Data

In [ ]:
class AEDDatatset(data.Dataset):
    def __init__(
        self,
        pathes: list[str | Path],
        labels: list[int] | None = None,
        augmentations: list | None = None,
        sr: int | None = None,
    ):
        self.pathes = pathes
        self.labels = labels
        if self.labels is not None:
            assert len(self.pathes) == len(self.labels), "Lenght error"
        self.sr = sr

        if augmentations is not None:
            self.augmentations = augmentations
        else:
            self.augmentations = lambda x: x
        self.stft = T.MelSpectrogram(self.sr)

    def __len__(self) -> tuple:
        return len(self.pathes)

    def __getitem__(self, idx: int) -> tuple:
        wav, _ = sf.read(self.pathes[idx])
        # [S] -> [B, C, S]
        wav = self.augmentations(wav, sample_rate=self.sr)
        wav = np.expand_dims(wav, axis=0)
        spec = (self.stft(torch.from_numpy(wav).to(torch.float32)) + 1e-5).log()
        if self.labels is not None:
            label = self.labels[idx]
            return spec, label
        return spec

In [ ]:
class AEDDataModule(pl.LightningDataModule):
    """AED DataModule for PyTorch Lightning."""

    def __init__(
        self,
        pathes: list[str, Path],
        labels: list[int],
        batch_size: int = 32,
        num_workers: int = 4,
        sr: int = 16_000,
        max_len: int = 32_000,
    ):
        super().__init__()
        self.pathes = pathes
        self.labels = labels

        self.batch_size = batch_size
        self.num_workers = num_workers
        self.sr = sr
        self.max_len = max_len

        # Define transforms
        self.train_augs = lambda x: x
        self.val_augs = A.Compose(
            [A.AdjustDuration(duration_samples=max_len, padding_mode="wrap", p=1.0), A.Normalize(p=1.0)]
        )

    def prepare_data(self):
        pass

    def setup(self, stage: str | None = None):
        if stage == "fit" or stage is None:
            pathes_train, pathes_val, labels_train, labels_val = train_test_split(
                self.pathes, self.labels, test_size=0.2, random_state=42, shuffle=True, stratify=self.labels
            )
            self.train_ds = AEDDatatset(pathes_train, labels_train, self.train_augs, sr=self.sr)
            self.val_ds = AEDDatatset(pathes_val, labels_val, self.val_augs, sr=self.sr)

    def __get_dataloader(self, ds: AEDDatatset, dl_type: str = "train"):
        return data.DataLoader(
            ds,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            pin_memory=True,
            shuffle=dl_type == "train",
            drop_last=dl_type == "train",
        )

    def train_dataloader(self):
        return self.__get_dataloader(self.train_ds, "train")

    def val_dataloader(self):
        return self.__get_dataloader(self.val_ds, "val")

In [ ]:
class AEDDataModulev0(AEDDataModule):
    """AED DataModule for PyTorch Lightning."""

    def __init__(
        self,
        pathes: list[str, Path],
        labels: list[int],
        batch_size: int = 32,
        num_workers: int = 4,
        sr: int = 16_000,
        max_len: int = 32_000,
    ):
        super().__init__()
        self.train_augs = A.Compose(
            [
                A.AddBackgroundNoise(
                    sounds_path="/kaggle/input/datasets/nhattruongdev/musan-noise/musan/speech",
                    min_snr_db=3.0,
                    max_snr_db=30.0,
                    noise_transform=A.PolarityInversion(),
                    p=0.4,
                ),
                A.ApplyImpulseResponse(
                    ir_path="/kaggle/input/datasets/tuannguyenvananh/room-impulse-response-and-noise-database/RIRS_NOISES",
                    p=0.3,
                ),
                A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
                A.AdjustDuration(duration_samples=max_len, padding_mode="wrap", p=1.0),
                A.Normalize(p=1.0),
            ]
        )

In [ ]:
class AEDAugDataModule(AEDDataModule):
    def __init__(self, *args):
        super().__init__(*args)
        self.train_augs = A.Compose(
            [
                A.AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
                A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
                A.AdjustDuration(duration_samples=self.max_len, padding_mode="wrap", p=1.0),
                A.Normalize(p=1.0),
            ]
        )

In [ ]:
class AEDAugDataModulev2(AEDDataModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_augs = A.Compose(
            [
                A.AddBackgroundNoise(
                    sounds_path="/kaggle/input/datasets/nhattruongdev/musan-noise/musan/speech",
                    min_snr_db=3.0,
                    max_snr_db=30.0,
                    noise_transform=A.PolarityInversion(),
                    p=0.4,
                ),
                A.ApplyImpulseResponse(
                    ir_path="/kaggle/input/datasets/tuannguyenvananh/room-impulse-response-and-noise-database/RIRS_NOISES",
                    p=0.3,
                ),
                A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
                A.AdjustDuration(duration_samples=self.max_len, padding_mode="wrap", p=1.0),
                A.Normalize(p=1.0),
            ]
        )

In [ ]:
class AEDAugDataModulev3(AEDDataModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_augs = A.Compose(
            [
                A.AddBackgroundNoise(
                    sounds_path="/kaggle/input/datasets/nhattruongdev/musan-noise/musan/speech",
                    min_snr_db=3.0,
                    max_snr_db=30.0,
                    noise_transform=A.PolarityInversion(),
                    p=0.4,
                ),
                A.ApplyImpulseResponse(
                    ir_path="/kaggle/input/datasets/tuannguyenvananh/room-impulse-response-and-noise-database/RIRS_NOISES",
                    p=0.3,
                ),
                A.HighPassFilter(max_cutoff_freq=1000, p=0.2),
                A.LowPassFilter(max_cutoff_freq=5000, p=0.2),
                A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
                A.AdjustDuration(duration_samples=self.max_len, padding_mode="wrap", p=1.0),
                A.Normalize(p=1.0),
            ]
        )

In [ ]:
class AEDAugDataModulev4(AEDDataModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_augs = A.Compose(
            [
                A.ApplyImpulseResponse(
                    ir_path="/kaggle/input/datasets/tuannguyenvananh/room-impulse-response-and-noise-database/RIRS_NOISES",
                    p=0.3,
                ),
                A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
                A.AdjustDuration(duration_samples=self.max_len, padding_mode="wrap", p=1.0),
                A.Normalize(p=1.0),
            ]
        )

## Model

In [ ]:
class BaseLineModel(nn.Module):
    def __init__(self, sample_rate=16000, n_classes=41):
        super().__init__()

        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=10, kernel_size=3, padding=1)
        self.cnn3 = nn.Conv2d(in_channels=10, out_channels=3, kernel_size=3, padding=1)

        # self.features = EfficientNet.from_pretrained('efficientnet-b0')
        # use it as features
        #         for param in self.features.parameters():
        #             param.requires_grad = False

        self.lin1 = nn.Linear(1000, 333)

        self.lin2 = nn.Linear(333, 111)

        self.lin3 = nn.Linear(111, n_classes)

    def forward(self, x):
        x = F.relu(self.cnn1(x))
        x = F.relu(self.cnn3(x))

        x = self.features(x)

        x = x.view(x.shape[0], -1)
        x = F.relu(x)

        x = F.relu(self.lin1(x))
        x = F.relu(self.lin2(x))
        return self.lin3(x)

    def inference(self, x):
        x = self.forward(x)
        return F.softmax(x)


class AEDLightningModel(pl.LightningModule):
    """Simple CNN model for AED classification."""

    def __init__(
        self,
        learning_rate: float = 1e-3,
        n_classes: int = 41,
    ):
        super().__init__()

        # Network layers
        self.model = BaseLineModel(n_classes=n_classes)
        self.metrics = torchmetrics.MetricCollection(
            {
                "acc": torchmetrics.Accuracy("multiclass", num_classes=n_classes, average="macro"),
                "precision": torchmetrics.Precision("multiclass", num_classes=n_classes, average="macro"),
                "recall": torchmetrics.Recall("multiclass", num_classes=n_classes, average="macro"),
                "f1": torchmetrics.F1Score("multiclass", num_classes=n_classes, average="macro"),
            },
            prefix="val_",
        )
        self.criterion = nn.CrossEntropyLoss()
        self.learning_rate = learning_rate

    def forward(self, spec):
        return self.model(spec)

    def _shared_step(self, batch, step_type):
        specs, labels = batch
        logits = self(specs)
        loss = self.criterion(logits, labels)

        if step_type == "val":
            m_out = self.metrics(logits, labels)
            self.log_dict(m_out)
            self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        return self(batch)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
            },
        }

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss, as described in https://arxiv.org/abs/1708.02002.

    It is essentially an enhancement to cross entropy loss and is
    useful for classification tasks when there is a large class imbalance.
    x is expected to contain raw, unnormalized scores for each class.
    y is expected to contain class labels.

    Shape:
        - x: (batch_size, C) or (batch_size, C, d1, d2, ..., dK), K > 0.
        - y: (batch_size,) or (batch_size, d1, d2, ..., dK), K > 0.
    """

    def __init__(
        self, alpha: Tensor | None = None, gamma: float = 0.0, reduction: str = "mean", ignore_index: int = -100
    ):
        """Constructor.

        Args:
            alpha (Tensor, optional): Weights for each class. Defaults to None.
            gamma (float, optional): A constant, as described in the paper.
                Defaults to 0.
            reduction (str, optional): 'mean', 'sum' or 'none'.
                Defaults to 'mean'.
            ignore_index (int, optional): class label to ignore.
                Defaults to -100.
        """
        if reduction not in ("mean", "sum", "none"):
            raise ValueError('Reduction must be one of: "mean", "sum", "none".')

        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction

        self.nll_loss = nn.NLLLoss(weight=alpha, reduction="none", ignore_index=ignore_index)

    def __repr__(self):
        arg_keys = ["alpha", "gamma", "ignore_index", "reduction"]
        arg_vals = [self.__dict__[k] for k in arg_keys]
        arg_strs = [f"{k}={v!r}" for k, v in zip(arg_keys, arg_vals)]
        arg_str = ", ".join(arg_strs)
        return f"{type(self).__name__}({arg_str})"

    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        if x.ndim > 2:
            # (N, C, d1, d2, ..., dK) --> (N * d1 * ... * dK, C)
            c = x.shape[1]
            x = x.permute(0, *range(2, x.ndim), 1).reshape(-1, c)
            # (N, d1, d2, ..., dK) --> (N * d1 * ... * dK,)
            y = y.view(-1)

        unignored_mask = y != self.ignore_index
        y = y[unignored_mask]
        if len(y) == 0:
            return torch.tensor(0.0)
        x = x[unignored_mask]

        # compute weighted cross entropy term: -alpha * log(pt)
        # (alpha is already part of self.nll_loss)
        log_p = F.log_softmax(x, dim=-1)
        ce = self.nll_loss(log_p, y)

        # get true class column from each row
        all_rows = torch.arange(len(x))
        log_pt = log_p[all_rows, y]

        # compute focal term: (1 - pt)^gamma
        pt = log_pt.exp()
        focal_term = (1 - pt) ** self.gamma

        # the full loss: -alpha * ((1 - pt)^gamma) * log(pt)
        loss = focal_term * ce

        if self.reduction == "mean":
            loss = loss.mean()
        elif self.reduction == "sum":
            loss = loss.sum()

        return loss


class FocalLossv2(nn.Module):
    def __init__(self, gamma=2, alpha=None, reduction="mean", task_type="binary", num_classes=None):
        """
        Unified Focal Loss class for binary, multi-class, and multi-label classification tasks.
        :param gamma: Focusing parameter, controls the strength of the modulating factor (1 - p_t)^gamma
        :param alpha: Balancing factor, can be a scalar or a tensor for class-wise weights. If None, no class balancing.
        :param reduction: Specifies the reduction method: 'none' | 'mean' | 'sum'
        :param task_type: Specifies the type of task: 'binary', 'multi-class', or 'multi-label'
        :param num_classes: Number of classes (only required for multi-class classification)
        """
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
        self.task_type = task_type
        self.num_classes = num_classes

        # Handle alpha for class balancing in multi-class tasks
        if task_type == "multi-class" and alpha is not None and isinstance(alpha, (list, torch.Tensor)):
            assert num_classes is not None, "num_classes must be specified for multi-class classification"
            if isinstance(alpha, list):
                self.alpha = torch.Tensor(alpha)
            else:
                self.alpha = alpha

    def forward(self, inputs, targets):
        """
        Forward pass to compute the Focal Loss based on the specified task type.
        :param inputs: Predictions (logits) from the model.
                       Shape:
                         - binary/multi-label: (batch_size, num_classes)
                         - multi-class: (batch_size, num_classes)
        :param targets: Ground truth labels.
                        Shape:
                         - binary: (batch_size,)
                         - multi-label: (batch_size, num_classes)
                         - multi-class: (batch_size,)
        """
        return self.multi_class_focal_loss(inputs, targets)

    def multi_class_focal_loss(self, inputs, targets):
        """Focal loss for multi-class classification."""
        if self.alpha is not None:
            alpha = self.alpha.to(inputs.device)

        # Convert logits to probabilities with softmax
        probs = F.softmax(inputs, dim=1)

        # One-hot encode the targets
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).float()

        # Compute cross-entropy for each class
        ce_loss = -targets_one_hot * torch.log(probs)

        # Compute focal weight
        p_t = torch.sum(probs * targets_one_hot, dim=1)  # p_t for each sample
        focal_weight = (1 - p_t) ** self.gamma

        # Apply alpha if provided (per-class weighting)
        if self.alpha is not None:
            alpha_t = alpha.gather(0, targets)
            ce_loss = alpha_t.unsqueeze(1) * ce_loss

        # Apply focal loss weight
        loss = focal_weight.unsqueeze(1) * ce_loss

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


class AEDFocalLightningModel(AEDLightningModel):
    def __init__(self, weights=torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = FocalLoss(weights)


class AEDFocalv2LightningModel(AEDLightningModel):
    def __init__(self, weights=torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = FocalLossv2(gamma=2, alpha=weights, task_type="multi-class", num_classes=len(label2idx))

## Training

In [ ]:
if CURR_RUN_TYPE in (RunType.BASELINE, RunType.AUGS, RunType.AUGSv2, RunType.AUGSv3, RunType.AUGSv4, RunType.FOCAL):
    pl.seed_everything(42, verbose=False)

    if CURR_RUN_TYPE == RunType.FOCAL:
        weights = [v for _, v in sorted(Counter(labels).items(), key=lambda x: x[0])]
        total_sum = sum(weights)
        weights = torch.tensor([v / total_sum for v in weights])
        # weights = torch.tensor(weights).to(torch.float32).softmax(-1)
        # wieghts = [1.0] * len(label2idx)
        lm = AEDFocalv2LightningModel(weights=weights, n_classes=len(label2idx))
    else:
        lm = AEDLightningModel(n_classes=len(label2idx))

    if CURR_RUN_TYPE == RunType.BASELINE:
        dm = AEDDataModule(pathes, labels, batch_size=32)
    elif CURR_RUN_TYPE == RunType.AUGS:
        dm = AEDAugDataModule(pathes, labels, batch_size=32)
    elif RunType.AUGSv2 == CURR_RUN_TYPE:
        dm = AEDAugDataModulev2(pathes, labels, batch_size=32)
    elif RunType.AUGSv3 == CURR_RUN_TYPE:
        dm = AEDAugDataModulev3(pathes, labels, batch_size=32)
    elif RunType.AUGSv4 == CURR_RUN_TYPE or CURR_RUN_TYPE == RunType.FOCAL:
        dm = AEDAugDataModulev4(pathes, labels, batch_size=32)
    trainer.fit(lm, datamodule=dm)

    sample_df = pd.read_csv("/kaggle/input/competitions/itmo-acoustic-event-detectin-2026/sample_submission.csv")
    pathes, labels, _ = get_pathes_labels_conv(sample_df, label2idx, pred=True)
    pred_dl = data.DataLoader(
        AEDDatatset(pathes, labels, dm.val_augs, dm.sr),
        num_workers=4,
        pin_memory=True,
        shuffle=False,
        batch_size=32,
    )
    preds = trainer.predict(lm, pred_dl, ckpt_path="best")

    preds = torch.cat(preds, dim=0)
    preds = preds.argmax(dim=-1)
    sample_df["label"] = preds.numpy()
    sample_df["label"] = sample_df["label"].apply(lambda idx: idx2label[idx])
    sample_df.to_csv("submit_augs_v4_focal_w.csv", index=False)

# EAT

## Data

In [ ]:
class EATDataset(data.Dataset):
    def __init__(
        self,
        pathes: list[str | Path],
        labels: list[int] | None = None,
        augmentations: list | None = None,
        sr: int | None = None,
    ):
        self.pathes = pathes
        self.labels = labels
        if self.labels is not None:
            assert len(self.pathes) == len(self.labels), "Lenght error"

        if augmentations is not None:
            self.augmentations = augmentations
        else:
            self.augmentations = lambda x: x

        self.target_length = 1024
        self.norm_mean = -4.268
        self.norm_std = 4.569

    def __len__(self) -> tuple:
        return len(self.pathes)

    def __getitem__(self, idx: int) -> tuple:
        wav, sr = sf.read(self.pathes[idx])

        waveform = torch.tensor(wav).float()
        if sr != 16_000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)

        waveform_np = waveform.numpy()
        waveform_np = self.augmentations(waveform_np, sample_rate=16_000)
        waveform = torch.tensor(waveform_np).float()

        # Normalize and convert to mel-spectrogram
        waveform = waveform - waveform.mean()
        mel = torchaudio.compliance.kaldi.fbank(
            waveform.unsqueeze(0),
            htk_compat=True,
            sample_frequency=16000,
            use_energy=False,
            window_type="hanning",
            num_mel_bins=128,
            dither=0.0,
            frame_shift=10,
        ).unsqueeze(0)

        # Pad or truncate
        n_frames = mel.shape[1]
        if n_frames < self.target_length:
            mel = torch.nn.ZeroPad2d((0, 0, 0, self.target_length - n_frames))(mel)
        else:
            mel = mel[:, : self.target_length, :]

        # Normalize
        mel = (mel - self.norm_mean) / (self.norm_std * 2)
        # mel = mel.unsqueeze(0)  # shape: [1, 1, T, F]

        if self.labels is not None:
            label = self.labels[idx]
            return mel, label
        return mel


class EATDataModule(AEDDataModule):
    """EAT DataModule for PyTorch Lightning."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_len = 16_000 * 10
        self.train_augs = A.Compose(
            [
                A.ApplyImpulseResponse(
                    ir_path="/kaggle/input/datasets/tuannguyenvananh/room-impulse-response-and-noise-database/RIRS_NOISES",
                    p=0.3,
                ),
                A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
            ]
        )

    def setup(self, stage: str | None = None):
        # Assign train/val datasets for use in dataloaders
        if stage == "fit" or stage is None:
            pathes_train, pathes_val, labels_train, labels_val = train_test_split(
                self.pathes, self.labels, test_size=0.2, random_state=42, shuffle=True, stratify=self.labels
            )
            self.train_ds = EATDataset(pathes_train, labels_train, self.train_augs, sr=self.sr)
            self.val_ds = EATDataset(pathes_val, labels_val, self.val_augs, sr=self.sr)

## Model

In [ ]:
class AttentiveProbe(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, num_layers: int = 4, num_heads: int = 4) -> None:
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim, nhead=num_heads, batch_first=True, norm_first=True, activation=F.silu
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(input_dim, num_classes)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        assert len(features.shape) == 3, "Input should be [batch, seq_len, hidden] size"
        out = self.transformer(features)
        out = out.mean(dim=1)
        return self.classifier(out)


class EfficientAudioHead(nn.Module):
    def __init__(self, input_dim: int, num_classes: int) -> None:
        super().__init__()

        # 1. Downsample the sequence length (1024 frames is too long for frozen features)
        # Strided Conv1d compresses time, saving memory and blending local frames
        self.conv_downsample = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=input_dim, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(input_dim),
            nn.SiLU(),
            nn.Conv1d(in_channels=input_dim, out_channels=input_dim // 2, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(input_dim // 2),
            nn.SiLU(),
        )

        mid_dim = input_dim // 2

        # 2. Attentive Pooling (instead of simple mean pooling)
        # This teaches the model to look for *where* the audio event happens
        self.attention_query = nn.Parameter(torch.randn(1, 1, mid_dim))
        self.mha = nn.MultiheadAttention(embed_dim=mid_dim, num_heads=4, batch_first=True)

        # 3. Classifer
        self.classifier = nn.Sequential(
            nn.Linear(mid_dim, mid_dim),
            nn.LayerNorm(mid_dim),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(mid_dim, num_classes),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        # features shape: [B, Seq, Hidden] (e.g., [32, 101, 768])

        # Prepare for Conv1d: [B, Hidden, Seq]
        x = features.transpose(1, 2)
        x = self.conv_downsample(x)
        x = x.transpose(1, 2)  # Back to [B, Reduced_Seq, Mid_Dim]

        # Apply Multihead Attention pooling using a learnable query token
        batch_size = x.shape[0]
        query = self.attention_query.expand(batch_size, -1, -1)

        # attn_out shape: [B, 1, Mid_Dim]
        attn_out, _ = self.mha(query, x, x)
        attn_out = attn_out.squeeze(1)  # [B, Mid_Dim]

        return self.classifier(attn_out)


class EATLightningModel(AEDLightningModel):
    """EAT model for AED classification."""

    def __init__(
        self,
        learning_rate: float = 1e-4,
        n_classes: int = 41,
    ):
        super().__init__(learning_rate=learning_rate, n_classes=n_classes)

        # Network layers
        model_id = "worstchan/EAT-base_epoch30_finetune_AS2M"
        # model_id = "worstchan/EAT-base_epoch30_pretrain"
        self.model = AutoModel.from_pretrained(model_id, trust_remote_code=True).eval()
        for param in self.model.parameters():
            param.requires_grad = False

        eat_output_dim = 768
        # self.head = AttentiveProbe(eat_output_dim, n_classes, 4, 8)
        self.head = EfficientAudioHead(eat_output_dim, n_classes)
        # self.head = nn.Sequential(
        #     nn.LayerNorm(eat_output_dim),
        #     nn.Linear(eat_output_dim, n_classes)
        # )

    @override
    def forward(self, spec):
        with torch.no_grad():
            feat = self.model.extract_features(spec)

        # [B, Seq, Hidden] -> [B, Hidden]
        # if len(feat.shape) == 3:
        #     feat = feat.mean(dim=1)  # [B, Seq, Hidden] -> [B, Hidden]
        # else:
        #     feat = feat.mean(dim=0)
        preds = self.head(feat)
        assert len(preds.shape) == 2, f"{preds.shape=}"
        return preds

    @override
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

## Training

In [ ]:
if CURR_RUN_TYPE in (RunType.BASESSL, "foo"):
    pl.seed_everything(42, verbose=False)

    lm = EATLightningModel(n_classes=len(label2idx))
    dm = EATDataModule(pathes, labels, batch_size=32)

    trainer.fit(lm, dm)

    torch.save(lm.head.state_dict(), "baseline_head.pth")

    sample_df = pd.read_csv("/kaggle/input/competitions/itmo-acoustic-event-detectin-2026/sample_submission.csv")
    pathes_pred, labels_pred, _ = get_pathes_labels_conv(sample_df, label2idx, pred=True)
    pred_dl = data.DataLoader(
        EATDataset(pathes_pred, labels_pred, dm.val_augs, dm.sr),
        num_workers=4,
        pin_memory=True,
        shuffle=False,
        batch_size=32,
    )
    preds = trainer.predict(lm, pred_dl, ckpt_path="best")
    preds = torch.cat(preds, dim=0)
    preds = preds.argmax(dim=-1)
    sample_df["label"] = preds.numpy()
    sample_df["label"] = sample_df["label"].apply(lambda idx: idx2label[idx])
    sample_df.to_csv("submit_eat_pretrain.csv", index=False)

# AST

## Data

In [ ]:
class ASTDataset(data.Dataset):
    def __init__(
        self,
        pathes: list[str | Path],
        labels: list[int] | None = None,
        augmentations: list | None = None,
        sr: int | None = None,
    ):
        self.pathes = pathes
        self.labels = labels
        if self.labels is not None:
            assert len(self.pathes) == len(self.labels), "Lenght error"

        if augmentations is not None:
            self.augmentations = augmentations
        else:
            self.augmentations = None

        self.feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

    def __len__(self) -> tuple:
        return len(self.pathes)

    def __getitem__(self, idx: int) -> tuple:
        wav, sr = sf.read(self.pathes[idx])
        if self.augmentations is not None:
            wav = self.augmentations(wav, sample_rate=sr)
        feats = self.extract_feature(wav, sr)

        if self.labels is not None:
            label = self.labels[idx]
            return feats, label

        return feats

    def extract_feature(self, wav, sr):
        x, _ = librosa.effects.trim(wav)
        return self.feature_extractor(x, sampling_rate=sr, return_tensors="pt")["input_values"]

## Model

In [ ]:
class ASTClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.pretrained_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593").eval()
        for param in self.pretrained_model.parameters():
            param.requires_grad = False

        self.head = nn.Sequential(
            nn.LayerNorm(768),
            nn.SiLU(),
            nn.Dropout(p=0.3),
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.SiLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = x.squeeze(1)
        with torch.no_grad():
            x = self.pretrained_model(input_values=x)
        if hasattr(x, "pooler_output") and x.pooler_output is not None:
            x = x.pooler_output
        else:
            x = x.last_hidden_state[:, 0, :]

        return self.head(x)


class ASTLightningModel(AEDLightningModel):
    """AST model for AED classification."""

    def __init__(
        self,
        learning_rate: float = 1e-4,
        n_classes: int = 41,
    ):
        super().__init__(learning_rate=learning_rate, n_classes=n_classes)
        self.model = ASTClassifier(n_classes)

    @override
    def forward(self, spec):
        return self.model(spec)

    @override
    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate, betas=(0.9, 0.99), eps=1e-06, fused=True)

## Training

In [ ]:
if CURR_RUN_TYPE == RunType.AST:
    pl.seed_everything(42, verbose=False)

    lm = ASTLightningModel(n_classes=len(label2idx))
    pathes_train, pathes_val, labels_train, labels_val = train_test_split(
        pathes, labels, test_size=0.2, random_state=42, shuffle=True, stratify=labels
    )
    train_augs = A.Compose(
        [
            A.ApplyImpulseResponse(
                ir_path="/kaggle/input/datasets/tuannguyenvananh/room-impulse-response-and-noise-database/RIRS_NOISES",
                p=0.3,
            ),
            A.PitchShift(min_semitones=-4, max_semitones=4, p=0.5),
        ]
    )
    train_ds = ASTDataset(pathes_train, labels_train, train_augs)
    train_dl = data.DataLoader(
        train_ds,
        batch_size=32,
        num_workers=4,
        pin_memory=True,
        shuffle=True,
        drop_last=True,
    )
    val_ds = ASTDataset(pathes_val, labels_val)
    val_dl = data.DataLoader(
        val_ds,
        batch_size=32,
        num_workers=4,
        pin_memory=True,
        shuffle=False,
        drop_last=False,
    )

    trainer.fit(lm, train_dl, val_dl)

    torch.save(lm.model.head.state_dict(), "baseline_head.pth")

    sample_df = pd.read_csv("/kaggle/input/competitions/itmo-acoustic-event-detectin-2026/sample_submission.csv")
    pathes_pred, labels_pred, _ = get_pathes_labels_conv(sample_df, label2idx, pred=True)
    pred_dl = data.DataLoader(
        ASTDataset(pathes_pred, labels_pred),
        num_workers=4,
        pin_memory=True,
        shuffle=False,
        batch_size=32,
    )
    preds = trainer.predict(lm, pred_dl, ckpt_path="best")

    preds = torch.cat(preds, dim=0)
    preds = preds.argmax(dim=-1)
    sample_df["label"] = preds.numpy()
    sample_df["label"] = sample_df["label"].apply(lambda idx: idx2label[idx])
    sample_df.to_csv("submit_ast_pretrain.csv", index=False)

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /kaggle/working/checkpoints/RunType.AST exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ASTClassifier    │ 86.4 M │ train │     0 │
│ 1 │ metrics   │ MetricCollection │      0 │ train │     0 │
│ 2 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 242 K                                                                                            
Non-trainable params: 86.2 M                                                                                       
Total params: 86.4 M                                                                                               
Total estimated model params size (MB): 345                                                                        
Modules in train mode: 19                                                                                          
Modules in eval mode: 212                                                                                          
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/loops/fit_loop.py:534: Found 212 module(s) in eval mode 
at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can 
ignore this warning.

Restoring states from the checkpoint path at /kaggle/working/checkpoints/RunType.AST/aed-epoch=16-val_f1=0.77.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /kaggle/working/checkpoints/RunType.AST/aed-epoch=16-val_f1=0.77.ckpt


Output()